# 트랜스포머 아키텍쳐 구현
lms를 참고하여 코드를 학습하고 마크다운과 주석을 달아본다.

## 1. 시각화 환경 설정

그래프에서 한글이 깨지지 않도록 matplotlib의 기본 폰트를 설정한다.

In [1]:
# 라이브러리 호출
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 그래프의 해상도 설정
%config InlineBackend.figure_format = 'retina'

# windows 기본 한글 폰트 경로
fontpath = 'C:/Windows/Fonts/malgun.ttf'
font = fm.FontProperties(fname=fontpath, size=9)

# matplotlib 기본 폰트를 한글 폰트로 설정
plt.rc('font', family=font.get_name())
mpl.font_manager.findfont(font)

# 그래프에서 마이너스 기호가 깨지는 것을 방지
mpl.rcParams['axes.unicode_minus'] = False

print("슝=3")

ModuleNotFoundError: No module named 'matplotlib'

## 2. 라이브러리 불러오기

In [ ]:
# 숫자 계산과 텐서 연산에 사용할 라이브러리
import numpy as np
import torch

# 신경망 모델을 구성하는 데 사용할 PyTorch 모듈
import torch.nn as nn

# activation, loss 등 함수 형태 기능을 사용하기 위한 모듈
import torch.nn.functional as F

# 모델 학습 시 파라미터 업데이트에 사용할 optimizer
import torch.optim as optim

# 그래프 시각화에 사용할 라이브러리
import matplotlib.pyplot as plt
import seaborn

# 파일 처리, 시간 측정, 랜덤 제어 등에 사용할 보조 라이브러리
import re
import os
import io
import time
import random
import math



print(torch.__version__)

## 3. Positional Encoding
Transformer는 입력을 한 번에 처리하기 때문에 토큰의 순서를 직접 알 수 없다. 따라서 각 위치를 나타내는 벡터를 만들어 단어 임베딩에 더해준다.
이 함수는 '(pos, d_model)' 크기의 위치 인코딩 테이블을 만든다.

In [ ]:
# 위치 정보 함수
def positional_encoding(pos, d_model):
    
    # 위치 position의 차원 i 값 계산
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)
    
    # 위치 position의 모든 차원 계산
    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]
    
    # 모든 위치의 벡터를 모아 2차원 테이블 생성 
    sinusoid_table = np.array(
        [get_posi_angle_vec(pos_i) for pos_i in range(pos)]
    )
    
    # 짝수 차원 sin, 홀수 차원 cos
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

print("슝=3")

## 4. Multi-Head Attention

In [ ]:
# 여러 head로 attention을 계산하는 모듈
class MultiHeadAttention(nn.Module):
    
    # Multi-Head Attention에 필요한 설정값과 Linear layer 준비
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # head 하나가 맡을 차원 수
        self.depth = d_model // self.num_heads

        # Q/K/V 변환용 Linear layer
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.linear = nn.Linear(d_model, d_model)

    # Q, K, V로 attention 계산
    def scaled_dot_product_attention(self, Q, K, V, mask):
        d_k = K.size(-1)

        # Q와 K의 유사도 계산
        QK = torch.matmul(Q, K.transpose(-2, -1))

        # sqrt(d_k)로 점수 크기 조절
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 볼 수 없는 위치 제거
        if mask is not None:
            scaled_qk += (mask * -1e9)

        # attention weight로 V를 섞음
        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)

        return out, attentions
    
    # Q/K/V를 head별로 나누는 함수
    def split_heads(self, x):
        batch_size = x.size(0)

        # d_model 차원을 num_heads와 depth로 reshape
        x = x.view(batch_size, -1, self.num_heads, self.depth)

        # attention 계산을 위해 head 차원을 앞으로 이동
        x = x.permute(0, 2, 1, 3)

        return x

    # head별로 나뉜 결과를 다시 합치는 함수
    def combine_heads(self, x):
        batch_size = x.size(0)

        # head 차원을 seq 뒤로 이동
        x = x.permute(0, 2, 1, 3)

        # 메모리 배치 정리 후 d_model로 복원
        x = x.contiguous().view(batch_size, -1, self.d_model)

        return x

    # 전체 계산 흐름
    def forward(self, Q, K, V, mask=None):
        # Linear 변환
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        # head 나누기
        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # attention 계산
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 합치기
        out = self.combine_heads(out)
        # 최종 Linear
        out = self.linear(out)

        return out, attention_weights


## 